# Vol6 Crypto Executable Pins

This is a **runnable subset** of `Vol6_crypto_novel_pins.ipynb`.

Policy:
- keep only cells that execute in a clean Python environment
- patch `matplotlib` import (pre-import `matplotlib.artist`)
- wrap heavy demo loops behind `RUN_DEMOS=False`

Kept 16 code cells; quarantined 8.


In [ ]:
RUN_DEMOS = False  # set True to run guarded demo cells


In [ ]:
RUN_DEMOS = RUN_DEMOS if 'RUN_DEMOS' in globals() else False
if RUN_DEMOS:
    # SOURCE: MergedNotebook.ipynb#c274
    import hashlib
    import numpy as np

    # Parameters
    base_message = b'genesis'
    delta_psi = 1.0
    k = 0.35  # damping factor for curvature
    threshold_H = 0.05
    threshold_Q = 0.95
    threshold_dpsi = 0.01
    max_iterations = 100000  # adjust as needed

    # Sigmoid function for non-linear curvature adjustment
    def sigmoid(x):
        return 1 / (1 + np.exp(-x))

    # Popcount: count bits set to 1 in byte array
    def bit_popcount(byte_data):
        return sum(bin(b).count('1') for b in byte_data)

    # Main simulation loop
    for nonce in range(max_iterations):
        # Construct input and compute SHA-256 hash
        input_data = base_message + str(nonce).encode()
        digest = hashlib.sha256(input_data).digest()

        # Compute potential and actual entropy
        P_i = np.sum(np.frombuffer(input_data, dtype=np.uint8))
        A_i = bit_popcount(digest)

        if P_i == 0:
            continue  # avoid division by zero

        # Compute H and Q(H)
        H_i = A_i / P_i
        Q_i = 1 - abs(A_i / 256 - 0.35)

        # Update Δψ recursively
        delta_psi = (1 - k) * delta_psi + sigmoid(delta_psi)

        # Check for convergence criteria
        if abs(H_i - 0.35) < threshold_H and Q_i > threshold_Q and delta_psi < threshold_dpsi:
            print(f"FOUND STABLE NONCE: {nonce}")
            print(f"Input: {input_data}")
            print(f"Digest: {digest.hex()}")
            print(f"H = {H_i:.4f}, Q = {Q_i:.4f}, Δψ = {delta_psi:.4f}")
            break
    else:
        print("No stable configuration found within iteration limit.")


In [ ]:
RUN_DEMOS = RUN_DEMOS if 'RUN_DEMOS' in globals() else False
if RUN_DEMOS:
    # SOURCE: MergedNotebook.ipynb#c299
    import os
    import re
    import ast
    import hashlib
    import json
    import random
    from pathlib import Path
    from concurrent.futures import ProcessPoolExecutor, as_completed
    import multiprocessing
    from collections import Counter, defaultdict

    print(f"CPU cores: {multiprocessing.cpu_count()}")


In [ ]:
# SOURCE: MergedNotebook.ipynb#c313
def spiralize_file(filepath):
    """Spiralize any supported file with cluster awareness."""
    filepath = Path(filepath)
    
    try:
        content = filepath.read_text(encoding='utf-8', errors='ignore')
    except Exception as e:
        return None, str(e)
    
    if len(content.strip()) < 50:
        return None, "too short"
    
    ext = filepath.suffix.lower()
    is_code = ext in ['.py', '.pyx', '.pyi']
    
    if is_code:
        py_struct = extract_python_structure(content)
        concepts = python_to_weighted_concepts(py_struct, content)
    else:
        concepts = extract_weighted_concepts(content, is_code=False)
    
    if not concepts:
        for line in content.split('\n'):
            if line.strip() and len(line.strip()) > 10:
                concepts.append({
                    'text': line.strip()[:200],
                    'keywords': [],
                    'weight': 0.3,
                    'resonances': []
                })
                if len(concepts) >= 5:
                    break
    
    h_sig = compute_h_signature(content, concepts)
    
    spiral = {
        "source": str(filepath),
        "type": "code" if is_code else "text",
        "h_signature": h_sig,
        "hash": hashlib.sha256(content.encode()).hexdigest()[:16],
        "attractor_coverage": [],
        "cluster_resonances": [],
        "structure": {}
    }
    
    all_keywords = set()
    for c in concepts:
        all_keywords.update(c['keywords'])
    spiral["attractor_coverage"] = list(all_keywords & ATTRACTORS)
    
    for c in concepts:
        if c['resonances']:
            spiral["cluster_resonances"].extend(c['resonances'])
    
    if concepts:
        spiral["structure"]["SEED"] = {
            "text": concepts[0]['text'],
            "weight": concepts[0]['weight'],
            "keywords": concepts[0]['keywords']
        }
        
        for i, concept in enumerate(concepts[1:], 1):
            spiral["structure"][f"SPIRAL_{i}"] = {
                "contains": f"SPIRAL_{i-1}" if i > 1 else "SEED",
                "extends": concept['text'],
                "weight": concept['weight'],
                "keywords": concept['keywords']
            }
        
        top_texts = [c['text'] for c in concepts[:5]]
        spiral["structure"]["COLLAPSE"] = " → ".join(top_texts)
    
    return spiral, content

In [ ]:
# SOURCE: MergedNotebook.ipynb#c426
import matplotlib.artist  # Nexus patch: ensure matplotlib.artist exists
import os, glob, math, json, random, string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Point this at your transcript CSVs ---
SHA_TRACE = "/mnt/data/sha256_xray_trace_block0.csv"      # example transcript
FIXED_GEAR = "/mnt/data/pi_seed_fixedgear.csv"            # example “fixed constants” gear transcript
FLOAT_GEAR = "/mnt/data/pi_seed_floatgear.csv"            # example “floating constants” gear transcript

print("Found SHA trace:", os.path.exists(SHA_TRACE))
print("Found fixed gear:", os.path.exists(FIXED_GEAR))
print("Found float gear:", os.path.exists(FLOAT_GEAR))

In [ ]:
# SOURCE: MergedNotebook.ipynb#c427
# ---------- Helpers: parsing & bit views ----------

def parse_hex_df(df, reg_cols=list("abcdefgh")):
    out = df.copy()
    for c in reg_cols:
        if out[c].dtype == object:
            out[c] = out[c].apply(lambda x: int(x,16) if isinstance(x,str) and x.startswith("0x") else int(x))
    return out

def words_to_bits(words, bits_per_word=32):
    # words: (n, 8) uint32
    n, m = words.shape
    out = np.zeros((n, m*bits_per_word), dtype=np.uint8)
    for wi in range(m):
        w = words[:, wi].astype(np.uint32)
        for b in range(bits_per_word):
            shift = bits_per_word - 1 - b
            out[:, wi*bits_per_word + b] = (w >> shift) & 1
    return out

_pc = np.frompyfunc(lambda x: int(x).bit_count(), 1, 1)
def popcount_u32(arr):
    return _pc(arr.astype(np.uint32)).astype(np.int16)

def words_to_popcounts(words):
    # words: (n, 8) uint32 -> (n,8) popcounts
    pcs = np.zeros(words.shape, dtype=np.int16)
    for i in range(words.shape[1]):
        pcs[:, i] = popcount_u32(words[:, i])
    return pcs

def load_trace_csv(path, reg_cols=list("abcdefgh")):
    df = pd.read_csv(path)
    df = parse_hex_df(df, reg_cols=reg_cols)
    words = df[reg_cols].to_numpy(dtype=np.uint32)
    bits  = words_to_bits(words)
    pcs   = words_to_popcounts(words)
    return df, words, bits, pcs


In [ ]:
# SOURCE: MergedNotebook.ipynb#c442
# --- Ensemble leakage (if bits_ds exists) ---
if "bits_ds" in globals():
    def leakage_metric_dataset(bits_ds, k, train_frac=0.7, seed=0):
        N,T,D = bits_ds.shape
        k=int(k)
        rng = np.random.default_rng(seed)
        idx = np.arange(N); rng.shuffle(idx)
        n_train = int(N*train_frac)
        tr, te = idx[:n_train], idx[n_train:]
        Xtr = bits_ds[tr,:,:k].reshape(-1,k).astype(np.float64)*2-1
        Ytr = bits_ds[tr,:,k:].reshape(-1,D-k).astype(np.float64)*2-1
        Xte = bits_ds[te,:,:k].reshape(-1,k).astype(np.float64)*2-1
        Yte = bits_ds[te,:,k:].reshape(-1,D-k).astype(np.float64)*2-1

        Xtr_b = np.hstack([Xtr, np.ones((Xtr.shape[0],1))])
        Xte_b = np.hstack([Xte, np.ones((Xte.shape[0],1))])

        lam=1e-2
        XtX = Xtr_b.T @ Xtr_b
        W = np.linalg.solve(XtX + lam*np.eye(XtX.shape[0]), Xtr_b.T @ Ytr)
        pred = Xte_b @ W
        Yhat = np.sign(pred); Yhat[Yhat==0]=1
        acc = (Yhat==Yte).mean(axis=0)
        return float(acc.mean()), float(acc.max()), float(acc.mean()-0.5)

    rows=[]
    for k in ks:
        mean_acc, max_acc, above = leakage_metric_dataset(bits_ds, k)
        rows.append({"k_bits":k, "mean_acc":mean_acc, "max_acc":max_acc, "above_chance":above})
    leak_ens = pd.DataFrame(rows)

    plt.figure(figsize=(8,4))
    plt.plot(leak_ens['k_bits'], leak_ens['above_chance'], marker='o')
    plt.title("Ensemble leakage proxy vs projection size")
    plt.xlabel("k (bits kept)")
    plt.ylabel("mean(acc)-0.5")
    plt.tight_layout()
    display(leak_ens)
else:
    print("bits_ds not defined; run the ensemble cell above if you want this.")


In [ ]:
# SOURCE: MergedNotebook.ipynb#c446
# --- Ensemble spectral gap proxy (if pcs_ds exists) ---
if "pcs_ds" in globals():
    def fit_transfer_popcount_ds(pcs_ds, k, center=True):
        S = pcs_ds[:, :, :k].astype(np.float64)
        X = S[:,:-1,:].reshape(-1,k)
        Y = S[:,1:,:].reshape(-1,k)
        if center:
            mu = X.mean(axis=0, keepdims=True)
            X = X - mu
            Y = Y - mu
        A_T = np.linalg.pinv(X) @ Y
        A = A_T.T
        s = np.linalg.svd(A, compute_uv=False)
        gamma = np.nan
        if len(s)>=2 and s[0]!=0:
            gamma = 1 - s[1]/s[0]
        return float(gamma), s

    rows=[]
    for k in [2,4,8]:
        gamma, s = fit_transfer_popcount_ds(pcs_ds, k)
        rows.append({"k_regs":k, "gamma":gamma, "s1":float(s[0]), "s2":float(s[1])})
    gap_ens = pd.DataFrame(rows)

    plt.figure(figsize=(6,4))
    plt.plot(gap_ens['k_regs'], gap_ens['gamma'], marker='o')
    plt.title("Ensemble spectral gap proxy γ vs projection size")
    plt.xlabel("k (registers kept)")
    plt.ylabel("γ = 1 - σ2/σ1")
    plt.tight_layout()
    display(gap_ens)
else:
    print("pcs_ds not defined; run the ensemble cell above if you want this.")


In [ ]:
# SOURCE: MergedNotebook.ipynb#c471
STOPWORDS = {
    # Articles, prepositions, conjunctions
    'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
    'of', 'with', 'by', 'from', 'as', 'is', 'was', 'are', 'were', 'been',
    'be', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 
    'could', 'should', 'may', 'might', 'must', 'shall', 'can', 'need',
    'it', 'its', 'this', 'that', 'these', 'those', 'i', 'you', 'he', 
    'she', 'we', 'they', 'what', 'which', 'who', 'when', 'where', 'why',
    'how', 'all', 'each', 'every', 'both', 'few', 'more', 'most', 'other',
    'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so',
    'than', 'too', 'very', 'just', 'also', 'now', 'here', 'there', 'then',
    'if', 'else', 'while', 'about', 'into', 'through', 'during', 'before',
    'after', 'above', 'below', 'between', 'under', 'again', 'further',
    'once', 'any', 'because', 'until', 'against', 'being', 'having',
    'doing', 'their', 'them', 'his', 'her', 'him', 'my', 'your', 'our',
    'me', 'us', 'up', 'down', 'out', 'off', 'over', 'under', 'am',
    
    # Common code/doc noise
    'none', 'true', 'false', 'null', 'undefined', 'return', 'def', 'class',
    'import', 'from', 'self', 'args', 'kwargs', 'param', 'params',
    'example', 'examples', 'note', 'notes', 'see', 'todo', 'fixme',
    'file', 'files', 'path', 'name', 'value', 'values', 'type', 'types',
    'data', 'string', 'int', 'float', 'list', 'dict', 'set', 'tuple',
    'get', 'set', 'new', 'use', 'used', 'using', 'uses',
    
    # Generic verbs
    'make', 'made', 'take', 'takes', 'give', 'gives', 'given', 'go',
    'goes', 'going', 'come', 'comes', 'coming', 'see', 'sees', 'seen',
    'know', 'knows', 'known', 'think', 'thinks', 'thought', 'want',
    'wants', 'say', 'says', 'said', 'tell', 'tells', 'told', 'ask',
    'asks', 'asked', 'work', 'works', 'working', 'try', 'tries', 'tried',
    'like', 'likes', 'look', 'looks', 'looking', 'find', 'finds', 'found',
    'let', 'lets', 'put', 'puts', 'keep', 'keeps', 'begin', 'begins',
    'seem', 'seems', 'help', 'helps', 'show', 'shows', 'shown',
    
    # Numbers written out
    'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight',
    'nine', 'ten', 'first', 'second', 'third', 'last', 'next',
    
    # Filler words
    'really', 'actually', 'basically', 'simply', 'just', 'even',
    'still', 'already', 'always', 'never', 'often', 'sometimes',
    'usually', 'probably', 'maybe', 'perhaps', 'however', 'therefore',
    'thus', 'hence', 'well', 'much', 'many', 'lot', 'bit', 'thing',
    'things', 'something', 'anything', 'nothing', 'everything',
    'way', 'ways', 'time', 'times', 'case', 'cases', 'point', 'points',
}

print(f"Stopwords loaded: {len(STOPWORDS)} terms")

In [ ]:
# SOURCE: MergedNotebook.ipynb#c493
# --- Twin-Prime Capo (TPC) overlay ---
# mode: "A" (transpose), "B" (clamp), or "C" (alt-clamp)
# p    : an odd prime; twin is p+2 (assumed prime for the pairs we choose)
# a    : base residue anchor in [0, p-1] (can be derived from seed)

def tpc_index(n_k: int, k: int, mode: str, p: int, a: int, M: int) -> int:
    if mode == "A":
        # transpose: capo offset alternates between p and p+2
        offset = p if (k & 1) == 0 else (p + 2)
        return (n_k + offset) % M

    elif mode == "B":
        # residue clamp to {a, a+2} modulo p (shortened string)
        # project n_k to the nearest of the two residues
        r = n_k % p
        t0 = (n_k - ((r - a) % p))         # class a
        t1 = (n_k - ((r - (a + 2)) % p))   # class a+2
        # choose the closer projection (preserves local intervals)
        return (t0 if abs(t0 - n_k) <= abs(t1 - n_k) else t1) % M

    elif mode == "C":
        # alternating clamp: even steps to class a mod p, odd steps to class (a+2) mod (p+2)
        if (k & 1) == 0:
            r = n_k % p
            return (n_k - ((r - a) % p)) % M
        else:
            q = p + 2
            r = n_k % q
            return (n_k - ((r - (a + 2)) % q)) % M

    else:
        return n_k  # no-op


In [ ]:
RUN_DEMOS = RUN_DEMOS if 'RUN_DEMOS' in globals() else False
if RUN_DEMOS:
    # SOURCE: MergedNotebook.ipynb#c599
    import os
    import re
    import hashlib
    import json
    from pathlib import Path
    from concurrent.futures import ProcessPoolExecutor, as_completed
    import multiprocessing
    from collections import Counter

    print(f"CPU cores: {multiprocessing.cpu_count()}")


In [ ]:
# SOURCE: MergedNotebook.ipynb#c719
import hashlib
import numpy as np

def sha_bit_mirror(twin_mid=4.0, header=b'314', rounds=8):
    # Encode twin_mid first for corrected input stream
    input_data = str(twin_mid).encode() + header
    sha = hashlib.sha256(input_data)
    digest = sha.digest()
    # Bit mirror logic via XOR roll
    bits = np.unpackbits(np.frombuffer(digest, dtype=np.uint8))
    mirrored = bits ^ np.roll(bits, 1)
    h = np.sum(mirrored) / len(mirrored)  # H-ratio
    return h, digest.hex()

# Midpoint test injection
mids = [4.0, 6.0, 12.0]
h_vals = []

for mid in mids:
    h, digest = sha_bit_mirror(twin_mid=mid)
    h_vals.append(h)
    print(f"Midpoint: {mid}, H: {h:.3f}")
    if abs(h - 0.35) < 0.05:
        print(f"Resonance Lock Achieved at mid={mid}")
        break

print("H-vals:", h_vals)


In [ ]:
# SOURCE: MergedNotebook.ipynb#c770
import hashlib
import struct
import numpy as np

# ---------- 1) pin the question (provide your 64-byte block) ----------
def pack_words(words32):
    # words32: list of 16 unsigned 32-bit ints (W0..W15)
    return b"".join(struct.pack(">I", w & 0xffffffff) for w in words32)

# Example rails (edit W0..W15 to shape the question)
W = [
    0x14159265, 0x92653589, 0xebe6ad9a, 0x6d9aca76,   # phase + complement
    0x00000011, 0x00000013, 0xffffffee, 0xffffffec,   # twin primes + complements
    0x00000000, 0x00000000, 0x00000000, 0x00000000,   # torsion (E_perp lives here)
    0xdeadbeef, 0x01234567, 0x89abcdef, 0xfedcba98    # anchors
]
question_bytes = pack_words(W)  # 64 bytes

# ---------- 2) compile with SHA-256 ----------
digest = hashlib.sha256(question_bytes).digest()         # 32 bytes, 256 bits
rng = np.random.default_rng(int.from_bytes(digest, "big"))

# ---------- 3) map digest to a tiny MLP (no training) ----------
def make_mlp_from_digest(digest_bytes, in_dim=32, hid=32, out_dim=16):
    # use digest as seed to draw deterministic weights
    seed = int.from_bytes(digest_bytes, "big")
    rng = np.random.default_rng(seed)
    W1 = rng.normal(0, 0.25, size=(hid, in_dim))
    b1 = rng.normal(0, 0.25, size=(hid,))
    W2 = rng.normal(0, 0.25, size=(out_dim, hid))
    b2 = rng.normal(0, 0.25, size=(out_dim,))
    return (W1, b1, W2, b2)

W1,b1,W2,b2 = make_mlp_from_digest(digest, in_dim=32, hid=32, out_dim=16)

def forward_once(x):
    h = np.tanh(W1 @ x + b1)
    y = W2 @ h + b2
    return y, h

# ---------- 4) provide a stimulus (typed, small) ----------
# e.g., 32-dim normalized feature vector (your substrate)
stimulus = np.linspace(-1, 1, 32)   # placeholder; replace with domain features
y, h = forward_once(stimulus)

# ---------- 5) meters (H, kappa, echo power, delta) ----------
def bit_harmony(vec):
    # normalize activations to pseudo-bits: sign as 0/1; target ~0.35 ones
    bits = (vec > 0).astype(np.int32)
    return bits.mean()

def echo_power(vec, bins=16):
    # proxy: variance of histogram of activations after tanh
    hist, _ = np.histogram(vec, bins=bins, range=(-1,1))
    m = hist.mean()
    return ((hist - m)**2).mean()

H = bit_harmony(h)                     # want ~0.35
Pi = echo_power(h)                     # want flat/low after alignment
delta = np.mean(np.abs(np.sort(h) - np.sort(h)))  # trivial δ here (idempotent check)

gate_ok = (abs(H - 0.35) < 0.05) and (Pi < 5.0)   # tighten empirically per backend


In [ ]:
# SOURCE: MergedNotebook.ipynb#c783
import hashlib

def byte1_nonce_generator(seed_a=1, seed_b=4, limit=10000, memory_stack=None):
    """Generate nonces with recursive folding logic and optional memory stack."""
    stack = memory_stack[:] if memory_stack else [seed_a, seed_b]
    nonces = []
    for _ in range(limit):
        if len(stack) < 2:
            break
        A = stack[-2]
        B = stack[-1]
        # Difference length (binary length of B - A)
        C = len(bin(B - A)[2:]) if B - A > 0 else 1
        stack.append(C)            # push C
        stack.append(C)            # push C again (placeholder for later modification)
        Z = A + B
        stack.append(Z)            # push sum
        D = Z - B
        stack[-3] = D             # replace the first C with D (feedback adjustment)
        Y = Z + B
        stack.append(Y)            # push expanded sum
        X = len(stack)
        stack.append(X)            # push current stack length (compression trigger)
        total = sum(stack)
        compressed = len(bin(total)[2:])
        stack.append(compressed)   # push compressed length of sum of stack
        close_byte = A + B
        stack.append(close_byte)   # push closing byte (A+B) to complete the cycle
        nonces.append(stack[-1])   # record the latest byte as a nonce output
    return nonces, stack

# Example usage: generate Byte1 sequence (first few nonces) and final stack
nonces, final_stack = byte1_nonce_generator(limit=5)  # using default seeds 1 and 4
print("Generated nonces:", nonces)
print("Final stack state:", final_stack[-10:])  # show last 10 elements of stack


In [ ]:
# SOURCE: MergedNotebook.ipynb#c831
# sha_calibrate.py
# Deterministic SHA-256 nonce sweep → H-index, rolling variance, quiet bands.
# Artifacts: CSV + plots + summary JSON.

import matplotlib.artist  # Nexus patch: ensure matplotlib.artist exists
import json, math, hashlib
from typing import Tuple, List, Dict, Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- low-level bits ----------

def sha256_hex(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def msg_with_nonce(base: bytes, nonce: int) -> bytes:
    # fixed concatenation protocol: base:decimal_nonce
    return base + b":" + str(nonce).encode("utf-8")

def hex_to_bits(hex_str: str) -> np.ndarray:
    v = int(hex_str, 16)
    # MSB-first, length 256
    return np.array([(v >> (255 - i)) & 1 for i in range(256)], dtype=np.uint8)

def leading_zero_bits(bits: np.ndarray) -> int:
    z = 0
    for b in bits:
        if b == 0: z += 1
        else: break
    return z

def hamming_distance_bits(a: np.ndarray, b: np.ndarray) -> int:
    return int(np.bitwise_xor(a, b).sum())

def rolling_variance(x: pd.Series, window: int) -> pd.Series:
    return x.rolling(window=window, min_periods=window).var()

def contiguous_spans(mask: np.ndarray, start_nonce: int = 0) -> List[Tuple[int,int]]:
    spans = []
    n = len(mask); i = 0
    while i < n:
        if mask[i]:
            j = i
            while j+1 < n and mask[j+1]:
                j += 1
            spans.append((start_nonce + i, start_nonce + j))
            i = j + 1
        else:
            i += 1
    return spans

def save_plot(path: str):
    plt.tight_layout()
    plt.savefig(path, dpi=140, bbox_inches="tight")
    plt.close()

# ---------- H-index (delta-first) ----------

def sweep_calibrate(base_msg: str,
                    nonce_range: Tuple[int,int],
                    H_target: float = 0.35,
                    H_tolerance: float = 0.02,
                    ham_var_window: int = 33,
                    quiet_quantile: float = 0.20,
                    out_prefix: str = "calibrate") -> Dict[str, Any]:

    base = base_msg.encode("utf-8")
    start, end = nonce_range

    rows = []
    prev_bits = None
    for nonce in range(start, end+1):
        h = sha256_hex(msg_with_nonce(base, nonce))
        bits = hex_to_bits(h)
        lz = leading_zero_bits(bits)
        ham = float("nan") if prev_bits is None else hamming_distance_bits(bits, prev_bits)
        prev_bits = bits

        lz_norm = lz / 256.0
        drift_norm = (ham / 256.0) if not math.isnan(ham) else float("nan")
        denom = (lz_norm + drift_norm) if not math.isnan(drift_norm) else float("nan")
        H = (lz_norm / denom) if denom and not math.isnan(denom) and denom > 0 else float("nan")

        rows.append((nonce, h, lz, ham, lz_norm, drift_norm, H))

    df = pd.DataFrame(rows, columns=["nonce","hash_hex","lz_bits","ham_to_prev","lz_norm","drift_norm","H"])
    df["ham_var"] = rolling_variance(df["ham_to_prev"], window=ham_var_window)

    H_low, H_high = H_target - H_tolerance, H_target + H_tolerance
    df["in_H_band"] = df["H"].between(H_low, H_high)

    valid_var = df["ham_var"].dropna()
    quiet_thresh = valid_var.quantile(quiet_quantile) if len(valid_var) > 0 else float("inf")
    df["quiet"] = df["ham_var"] <= quiet_thresh

    both = (df["in_H_band"] & df["quiet"]).fillna(False).to_numpy()
    spans = contiguous_spans(both, start_nonce=start)

    csv_path = f"{out_prefix}_sha_phase_scan.csv"
    df.to_csv(csv_path, index=False)

    # H-index over nonce
    plt.figure(figsize=(10,3))
    plt.plot(df["nonce"], df["H"])
    plt.axhline(H_target, linestyle="--")
    plt.axhline(H_low, linestyle=":")
    plt.axhline(H_high, linestyle=":")
    save_plot(f"{out_prefix}_H_vs_nonce.png")

    # Hamming-to-previous
    plt.figure(figsize=(10,3))
    plt.plot(df["nonce"], df["ham_to_prev"])
    save_plot(f"{out_prefix}_hamming_vs_nonce.png")

    # Rolling variance
    plt.figure(figsize=(10,3))
    plt.plot(df["nonce"], df["ham_var"])
    save_plot(f"{out_prefix}_ham_var_vs_nonce.png")

    summary = {
        "run": "SHA_CALIBRATE",
        "base_msg": base_msg,
        "nonce_range": [start, end],
        "H_target": H_target,
        "H_tolerance": H_tolerance,
        "ham_var_window": ham_var_window,
        "quiet_quantile": quiet_quantile,
        "quiet_threshold": None if quiet_thresh == float("inf") else float(quiet_thresh),
        "band_spans": spans,
        "artifacts": {
            "csv": csv_path,
            "plots": [
                f"{out_prefix}_H_vs_nonce.png",
                f"{out_prefix}_hamming_vs_nonce.png",
                f"{out_prefix}_ham_var_vs_nonce.png",
            ]
        }
    }
    with open(f"{out_prefix}_summary.json","w") as f:
        json.dump(summary, f, indent=2)

    # Console certificate
    print(json.dumps({
        "ps": "calibrate-complete",
        "H_band": [H_low, H_high],
        "quiet_q": quiet_quantile,
        "quiet_threshold": summary["quiet_threshold"],
        "first_spans": spans[:5]
    }, indent=2))

    return summary

if __name__ == "__main__":
    # Defaults from your prompt
    sweep_calibrate(
        base_msg="PGGSPHRKCGYDLQNRGHPQW",
        nonce_range=(0, 50000),   # you can lower to (0, 20000) for a quick run
        H_target=0.35,
        H_tolerance=0.02,
        ham_var_window=33,
        quiet_quantile=0.20,
        out_prefix="calibrate"
    )

## Quarantined cells (did not run)
| cell | error | note |
|---:|---|---|
| 3 | NameError: name 'ATTRACTORS' is not defined | missing dependency / variable not defined in this subset |
| 7 | FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/sha256_xray_ | requires local file path |
| 8 | NameError: name 'fix_pcs' is not defined | missing dependency / variable not defined in this subset |
| 9 | NameError: name 'sha_pcs' is not defined | missing dependency / variable not defined in this subset |
| 10 | NameError: name 'sha_bits' is not defined | missing dependency / variable not defined in this subset |
| 12 | NameError: name 'sha_pcs' is not defined | missing dependency / variable not defined in this subset |
| 16 | NameError: name 'matrix' is not defined | missing dependency / variable not defined in this subset |
| 19 | NameError: name 'hex_spiral' is not defined | missing dependency / variable not defined in this subset |
